In [1]:

import torch

#detect the device available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device) # You may need to execute "pip install torch --index-url https://download.pytorch.org/whl/cu124" in order to use the GPU

Using device:  cuda


## Exercise 1: Prompt Engineering

Let's consider LLAMA as our starting point. In the following, we see a typical prompt feeding and text generation with LLAMA

In [10]:

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# HuggingFace token

#----------------Google Colab only---------------------
# from huggingface_hub import login
#from google.colab import userdata
#login(userdata.get('HF_TOKEN'))
#----------------Google Colab only---------------------

#----------------Local Runtime only---------------------
import os
from dotenv import load_dotenv # pip install python-dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HF_TOKEN") # Create this constant in the file .env (put it in the root dir of your project)
login(token=hf_token)
#----------------Local Runtime only---------------------


# Load the model and the tokenizer (the first time it requires a lot of time to be executed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "meta-llama/Llama-3.2-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map=device, token=hf_token)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, clean_up_tokenization_spaces=False)


# Configure generation parameters
gen_config = GenerationConfig.from_model_config(model.config)
# gen_config.temperature = 0.3              # Regulate randomness, use with do_sample = True
# gen_config.max_length = None              # Since max_new_tokens is already set
gen_config.repetition_penalty = 2.0         # Penalize repetition
gen_config.no_repeat_ngram_size = 3         # Avoid repeating bigrams
gen_config.do_sample = False                # Set to False in order to use Greedy or Beam search
gen_config.num_beams = 3                    # Use with do_sample = False
gen_config.eos_token_id = tokenizer.eos_token_id  # End generation at EOS token
gen_config.pad_token_id = tokenizer.pad_token_id
gen_config.early_stopping = True


query = "Tell me the capital of France."
messages = [
    {'role': 'user', 'content': query},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)



#alternative 1: use the tokenizer and produce the output

def make_a_query(_prompt: str, _tokenizer, _model, _device, _gen_config, max_new_tokens:int = 200):

    # Set pad_token_id if missing
    if not hasattr(_tokenizer, "pad_token_id") or _tokenizer.pad_token_id is None:
        _tokenizer.pad_token_id = _tokenizer.eos_token_id

    _gen_config.max_new_tokens = max_new_tokens  # Limit the number of new tokens generated (e.g., a single word)


    # Tokenize the input with padding and truncation
    inputs = _tokenizer(_prompt, return_tensors="pt", padding=True, truncation=True).to(_device)

    # Compute the length of the input prompt to be able to extract the model's response later
    input_ids = inputs["input_ids"]
    prompt_length = input_ids.shape[1]

    # Generate a response
    output = _model.generate(
        input_ids,
        attention_mask=inputs["attention_mask"],
        generation_config=_gen_config,
    )

    generated_tokens = output[0, prompt_length:]    # Try to print the variable "output" in order to understand its dimensionality

    # Decode the response into human-readable text
    response = _tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return response

print("\n----------------------------------------------\n\n")
print(make_a_query(_prompt=prompt, _tokenizer=tokenizer, _model=model, _gen_config=gen_config, _device=device))


#alternative 2: use the chat template and pipelines from huggingface

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    dtype=torch.float16,
    device_map="auto",
)

gen_config.max_new_tokens = 200

outputs = text_generation_pipeline(
    prompt,
    generation_config=gen_config,
) # Question: will this second generation be influenced by the previous one?

print("\n----------------------------------------------\n\n")
print(outputs[0]["generated_text"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell me the capital of France.<|eot_id|><|start_header_id|>assistant<|end_header_id|>



----------------------------------------------


The capital city of France is Paris.

----------------------------------------------


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell me the capital of France.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital city of France is Paris.


### Pymupdf

Reference libraries to install: pip install openai pymupdf faiss-cpu scikit-learn

PyMuPDF is a Python library that provides tools for working with PDF files (as well as other document formats like XPS, OpenXPS, CBZ, EPUB, and FB2). It's built on the MuPDF library, a lightweight, high-performance PDF and XPS rendering engine. With PyMuPDF, you can perform various tasks like reading, creating, editing, and extracting content from PDFs, images, and annotations.

In [3]:
#-----------Google Colab only----------------------
#!pip install pymupdf # The name of the module to be installed may vary depending on the environment where you run this code (i.e., Colab versus local python environment)
# Note: Ensure "example2.pdf" is uploaded to the Colab runtime before running this.
# If the file is missing, you will get a "RuntimeError: cannot open ..."
#--------------------------------------------------

import pymupdf

#open an example pdf
doc = pymupdf.open("example2.pdf")

# Extract text from the first page
page = doc.load_page(0)
text = page.get_text("text")  # Use 'text' mode to get raw text


def extract_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text+=page.get_text()
    return text

print(extract_text_from_pdf("example2.pdf"))

Associazione Calcio Milan, commonly referred to as AC Milan or simply Milan, is an Italian 
professional football club based in Milan, Lombardy. Founded in 1899, the club competes in the Serie 
A, the top tier of Italian football. In its early history, Milan played its home games in different grounds 
around the city before moving to its current stadium, the San Siro, in 1926. The stadium, which was 
built by Milan's second chairman, Piero Pirelli and has been shared with Inter Milan since 1947, is the 
largest in Italian football, with a total capacity of 75,817. The club has a long-standing rivalry with Inter, 
with whom they contest the Derby della Madonnina, one of the most followed derbies in football. 
 
Milan has spent its entire history in Serie A with the exception of the 1980–81 and 1982–83 seasons. 
Silvio Berlusconi’s 31-year tenure as Milan president was a standout period in the club's history, as 
they established themselves as one of Europe's most dominant and successful

### Example: Text Summarization

Let's ask LLAMA to perform a summarization of the example PDF.

In [4]:

# Define the prompt to ask for text summarization.
text_summarization_prompt = "Write a concise summary of the main ideas in article below."
text = extract_text_from_pdf("example2.pdf")                          #load here the text of the article
prompt_str = """{PROMPT}.\n\n article:\n {BODY}""".format(PROMPT=text_summarization_prompt, BODY=text)
messages = [
    {'role': 'user', 'content': prompt_str}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Feed the prompt to llama
print(make_a_query(prompt, tokenizer, model, device))

Here are the main points from the article about AC Milan:

* AC Milan is a professional football club founded in 1909 (corrected year) in Milano, Italy.
* The club plays in the Seria A, Italy's top tier, and shares its home stadium, Stadio Giuseppe Meazza (San Siro), with local rival Inter Milan.
* Milan has a rich history, having won numerous domestic and international titles, including 7 European Cup/UEFA Champions League trophies.
* Notable periods of success include Silvio Berusconi's 31 years as president, who led the team to 29 trophies, and the 1991-92 season when they went undefeated in the league.
* Domestic titles include 19 Scudetto (league titles), 5 Coppa Italia (Italian Cup) titles, and 4 Supercoppi Italiana (Italian Super Cup) awards.
* Internationally, Milan holds a record number of European Cup wins and has


### Adding a System Prompt

Llama was trained with a system message that set the context and persona to assume when solving a task. One of the unsung advantages of open-access models is that you have full control over the system prompt in chat applications. This is essential to specify the behavior of your chat assistant –and even imbue it with some personality– but it's unreachable in models served behind APIs.


In [5]:
#default standard system message from the Hugging Face blog to the prompt from above
system_prompt = "You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information."


# The same user prompt string as before
prompt_str = """{PROMPT}. article: {BODY}""".format(PROMPT=text_summarization_prompt, BODY=text)

# Put the system prompt and the user prompt in the same chat
messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': prompt_str}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(make_a_query(prompt, tokenizer, model, device))

The article provides an overview of AC Milan, a professional football club from Milan, Italy. Here are the main points:

**Club History and Stadium**: AC Milan was founded in 1869 and currently plays in the Seria A, Italy's top-tier league. The team moved to its iconic San Sira stadium in 1936, which it shares with rival Inter Milan.

**Success and Achievements**: Milan has a rich history of success, winning numerous domestic and international titles, including:

* 19 Serie A league titles
* 5 Copa Italia (Italian Cup)
* 7 Super Coppa Italiana (Italian Super Cup) trophies
* Record-breaking 7 European Cup/Champion's League titles

**Notable Players and Coaches**: The team has produced several notable players who have won individual awards, such as the Ballon D'Or. Notably, three of Milan's players (Marco van Bastein, Ruurd Gullitt, and Frans R


### Customizing the System prompt

With Llama we have full control over the system prompt. The following experiment will instruct Llama to assume the persona of a researcher tasked with writing a concise brief.

Apply the following changes the original system prompt:
- Use the researcher persona and specify the tasks to summarize articles.
- Remove safety instructions; they are unnecessary since we ask Llama to be truthful to the article.


In [6]:
new_system_prompt = """You are a helpful assistant.
    When summarizing an article or providing key points, you must:
    1. Use **bullet points** to list the main ideas.
    2. **Bullet points must be formatted like this:**
       - First key point goes here.
       - Second key point goes here.
       - Third key point goes here.
    3. Each bullet point should be one short sentence.
    4. Keep bullet points clear, concise, and easy to read.
    5. Make sure every distinct idea from the article is listed as a separate bullet point.
    6. Avoid paragraphs; only bullet points should be used in the output.
    7. Format bullet points (use bullet points exactly like this: - point 1)"""

# The same user prompt string as before
prompt_str = """{PROMPT}. article: {BODY}""".format(PROMPT=text_summarization_prompt, BODY=text)

messages = [
    {'role': 'system', 'content': new_system_prompt},
    {'role': 'user', 'content': prompt_str}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

r3 =  make_a_query(prompt, tokenizer, model, device)

print(r3)

- **Founding and History**: AC Milan is anItalian professional football club founded in 1889, competing in Serie Asince 1901.
- **Home Stadium**: The club plays at the iconicSan Siro stadium, sharing it withInter Milan, built byPiero Pirellisince1926.
-   **Rivalry**: Milan has a longstanding rivalry withInter Milansharing theDerby dellaMadonninaone of Italy'smost-followed derbies.
-     **Serie A Dominance**: Milanhas spent itsentire historyin Serie Aexcept for the1980-81 and1982-83seasons.
-      **Berlusconi Era**: SilvioBerlusconipresidency(1986-2017) led to29trophy wins, including multipleSerie ASeries andUEFA ChampionsLeague titles.
-    **International Success**: MilanisItaly'smost successfulclub, winning7European Cup/Championshipstitles,


### Chain-of-Thought prompting

Chain-of-thought is when a prompt is being constructed using a previous prompt answer. For our use case to extract information from text, we will first ask Llama what the article is about and then use the response to ask a second question: what problem does [what the article is about] solve?



In [7]:

new_system_prompt = """You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information.
    Provide straight to the point answers, without bullet points."""


# Now embed the result of the previous prompt in a new prompt to ask some additional information (i.e., the year of foundation). you have to split the original prompt to create a new one

additional_info_prompt = """How many titles did AC Milan win?"""

# The new user prompt contains a different request than before
prompt_str2 =  """{PROMPT}.\n\n article: \n{BODY}""".format(PROMPT=additional_info_prompt, BODY=r3)

messages = [
    {'role': 'system', 'content': new_system_prompt},
    {'role': 'user', 'content': prompt_str2}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


print("PROMPT:\n")
print(prompt)
print("\n\n\n")

print("RESPONSE:\n")
print(make_a_query(prompt, tokenizer, model, device))


PROMPT:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

You are a helpful, respectful and honest assistant.     Always answer as helpfully as possible, while being safe. Your answers should not include any harmful,     unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses     are socially unbiased and positive in nature. If a question does not make any sense, or is not factually     coherent, explain why instead of answering something not correct. If you don't know the answer to a question,     please don't share false information.
    Provide straight to the point answers, without bullet points.<|eot_id|><|start_header_id|>user<|end_header_id|>

How many titles did AC Milan win?.

 article: 
- **Founding and History**: AC Milan is anItalian professional football club founded in 1889, competing in Serie Asince 1901.
- **Home Stadium**: The club plays at the iconicSa

### Generating JSONs with Llama

Llama needs precise instructions when asking it to generate JSON. In essence, here is what works for me to get valid JSON consistently:

- Explicitly state — “ All output must be in valid JSON. Don’t add explanation beyond the JSON” in the system prompt.
- Add an “explanation” variable to the JSON example. Llama enjoys explaining its answers. Give it an outlet.
- Use the JSON as part of the instruction. See the “in_less_than_ten_words” example below.
Change “write the answer” to “output the answer.”


In [8]:


#example addition to a prompt to deal with jsons
json_prompt_addition = "Output must be in valid JSON like the following example {{\"sport\": sport, \"explanation\": [in_less_than_ten_words]}}. Output must include only JSON."

new_system_prompt = """
    You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information.
    Provide straight to the point answers, without bullet points.""" \
    + json_prompt_addition

text_summarization_prompt3 = "Summarize the article below. Provide the different main aspects of AC Milan in JSON format in the format of the above example. Do not add any fields with respect to the provided example."

#now generate a prompt by correctly concatenating the system prompt, the json prompt instruction, and a sports article
prompt_str3 = """{PROMPT}. \n\narticle: \n{BODY}""".format(PROMPT=text_summarization_prompt3, BODY=text)

messages = [
    {'role': 'system', 'content': new_system_prompt},
    {'role': 'user', 'content': prompt_str3}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("PROMPT:\n")
print(prompt_str3)

print("\n\n\n")

print("RESPONSE:\n")
print(make_a_query(prompt, tokenizer, model, device))

#compare the difference between the prompt with the formatting instruction and a regular prompt without formatting instructions. is there any difference?




PROMPT:

Summarize the article below. Provide the different main aspects of AC Milan in JSON format in the format of the above example. Do not add any fields with respect to the provided example.. 

article: 
Associazione Calcio Milan, commonly referred to as AC Milan or simply Milan, is an Italian 
professional football club based in Milan, Lombardy. Founded in 1899, the club competes in the Serie 
A, the top tier of Italian football. In its early history, Milan played its home games in different grounds 
around the city before moving to its current stadium, the San Siro, in 1926. The stadium, which was 
built by Milan's second chairman, Piero Pirelli and has been shared with Inter Milan since 1947, is the 
largest in Italian football, with a total capacity of 75,817. The club has a long-standing rivalry with Inter, 
with whom they contest the Derby della Madonnina, one of the most followed derbies in football. 
 
Milan has spent its entire history in Serie A with the exception of the

### One-to-Many Shot Learning Prompting

One-to-Many Shot Learning is a term that refers to a type of machine learning problem where the goal is to learn to recognize many different classes of objects from only one or a few examples of each class. For example, if you have only one image of a cat and one image of a dog, can you train a model to distinguish between cats and dogs in new images? This is a challenging problem because the model has to generalize well from minimal data (source)

Important points about the prompts:

- The system prompt includes the instructions to output the answer in JSON.
- The prompt consists of a one-to-many shot learning section that starts after the end of the system prompt.
- Shot examples are represented through pairs composed by user's question and assistant's response, as reported in the template below.
- The examples are given in JSON because the answers need to be JSON.
- The JSON allows defining the response with name, type, and explanation.
- The prompt question is represented by the last user's question - assistant's response pair, where the response is blank and the last <|eot_id|> is missing.

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Istruzioni generali sul comportamento dell’assistente.
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Question shot 1.
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer shot 1.
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Question shot 2.
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer shot 2.
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Final prompt.
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
```


In [9]:
new_system_prompt = """You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information.
    Provide straight to the point answers, without bullet points.
    Output must be a valid JSON array composed by multiple dictionaries; one dictionary for each term, by describing names and descriptions extracted from the article:

     [
        {"name": "term", "type": "category", "explanation": "definition"},
        {"name": "term2", "type": "category2", "explanation": "definition2"},
        ...
     ]
     """

shot_examples = [
    {'role': 'user', 'content': """Summarize the article below. List the different main aspects in VALID JSON, following the format of the above example.

**Article:**
The rapid growth of the semiconductor industry has transformed global manufacturing, as companies specializing in advanced chip design and microdevice fabrication struggle to meet soaring technology demands."""},
    {'role': 'assistant', 'content': """[
    {"name": "semiconductor", "type": "industry", "explanation": "Companies engaged in the design and fabrication of semiconductors and semiconductor devices"},
]"""},

    {'role': 'user', 'content': """**Article:**
The NBA continues to captivate millions of fans worldwide as the premier global stage for professional basketball, driving massive television ratings and international merchandise sales each season."""},
    {'role': 'assistant', 'content': """[
    {"name": "NBA", "type": "sport league", "explanation": "NBA is the national basketball league"}
]"""},

    {'role': 'user', 'content': """**Article:**
John Smith recently test-drove the new Ford F150, praising the latest engineering updates from the iconic American company, Ford, for setting a high standard in the modern pickup truck market."""},
    {'role': 'assistant', 'content': """[
    {"name": "Ford F150", "type": "vehicle", "explanation": "Article talks about the Ford F150 truck"},
    {"name": "Ford", "type": "company", "explanation": "Ford is a company that built vehicles"},
    {"name": "John Smith", "type": "person", "explanation": "Mentioned in the article"}
]"""},

]

text_summarization_prompt3 = "Summarize the article below. Provide the different main aspects using JSON format like the one of the above example."
text = extract_text_from_pdf("example2.pdf")

#now generate a prompt by correctly concatenating the system prompt, the json prompt instruction, and a sports article
prompt_str = """{PROMPT}. \n\n**Article:** \n{BODY}""".format(PROMPT=text_summarization_prompt3, BODY=text)

messages = [
    {'role': 'system', 'content': new_system_prompt},
    *shot_examples,
    {'role': 'user', 'content': prompt_str}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("PROMPT:\n")
print(prompt_str)

print("\n\n\n")

print("RESPONSE:\n")
print(make_a_query(prompt, tokenizer, model, device, max_new_tokens=400)) # The usual limit of 200 tokens may not be enough

#compare the difference between the prompt with the formatting instruction and a regular prompt without formatting instructions. is there any difference?



PROMPT:

Summarize the article below. Provide the different main aspects using JSON format like the one of the above example.. 

**Article:** 
Associazione Calcio Milan, commonly referred to as AC Milan or simply Milan, is an Italian 
professional football club based in Milan, Lombardy. Founded in 1899, the club competes in the Serie 
A, the top tier of Italian football. In its early history, Milan played its home games in different grounds 
around the city before moving to its current stadium, the San Siro, in 1926. The stadium, which was 
built by Milan's second chairman, Piero Pirelli and has been shared with Inter Milan since 1947, is the 
largest in Italian football, with a total capacity of 75,817. The club has a long-standing rivalry with Inter, 
with whom they contest the Derby della Madonnina, one of the most followed derbies in football. 
 
Milan has spent its entire history in Serie A with the exception of the 1980–81 and 1982–83 seasons. 
Silvio Berlusconi’s 31-year tenure 